# How to Use the "sel" Parameter for Datasets with >3 Dimensions

Datasets with more than the three spatio-temporal dimensions (e.g. x, y, time) will require the use of the `sel` parameter to pick a particular value of those non-spatio-temporal dimensions. The `sel` parameter can be used more than once in a given request. The `sel` parameter can also be used to specify a time within a single granule's data file.

This notebook demonstrates how to get statistics for a single time slice of a granule from the TROPESS O3 dataset, which also includes a fourth dimension `lev`. This dataset has annual granules each with dimensions for time (monthly). Queries within a single year will return the same granule. If `time={datetime}` is present in a sel query parameter value, titiler-cmr will pass the datetime query parameter value to the xarray `sel` parameter for the `time` dimension.

See also: [xarray.DataArray.sel](https://docs.xarray.dev/en/latest/generated/xarray.DataArray.sel.html).

## Setup

In [ ]:
import json
from datetime import datetime, timezone

import httpx

titiler_endpoint = "https://staging.openveda.cloud/api/titiler-cmr"  # staging endpoint

In [ ]:
geojson_dict = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {},
            "geometry": {
                "coordinates": [
                    [
                        [-20.79973248834736, 83.55979308678764],
                        [-20.79973248834736, 75.0115425216471],
                        [14.483337068956956, 75.0115425216471],
                        [14.483337068956956, 83.55979308678764],
                        [-20.79973248834736, 83.55979308678764],
                    ]
                ],
                "type": "Polygon",
            },
        }
    ],
}

r = httpx.post(
    f"{titiler_endpoint}/xarray/statistics",
    params=(
        ("collection_concept_id", "C2837626477-GES_DISC"),
        # Temporal query for CMR granule query
        ("temporal", datetime(2021, 10, 10, tzinfo=timezone.utc).isoformat()),
        # xarray backend query parameters
        ("variables", "o3"),
        ("sel", "time=nearest::{datetime}"),  #
        ("sel", "lev=1000"),
    ),
    json=geojson_dict,
    timeout=None,
).json()

print(json.dumps(r, indent=2))

You can chose a different time slice from the same granule simply by updating the `datetime` query parameter.

In [ ]:
r = httpx.post(
    f"{titiler_endpoint}/xarray/statistics",
    params=(
        ("collection_concept_id", "C2837626477-GES_DISC"),
        # Datetime for CMR granule query
        ("temporal", datetime(2021, 12, 10, tzinfo=timezone.utc).isoformat()),
        # xarray backend query parameters
        ("variables", "o3"),
        ("sel", "time=nearest::{datetime}"),  #
        ("sel", "lev=1000"),
    ),
    json=geojson_dict,
    timeout=None,
).json()

print(json.dumps(r, indent=2))